In [97]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY"):
    val = os.getenv(key)
    if val:
        os.environ[key] = val
        print(f"{key}: {val[:8]}...{val[-4:]}")
    else:
        print(f"{key} not found.")

os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")

OPENAI_API_KEY: sk-proj-...66gA
GOOGLE_API_KEY: AQ.Ab8RN...hGsw


'1'

In [ ]:
from agents import Agent, Runner
from google import genai
from google.genai import types

# OpenAI models for text agents
MODEL_TEXT = "gpt-4.1-nano"
MODEL_REVIEW = "gpt-4.1-mini"
MODEL_IMAGE_PROMPT = "gpt-4.1-mini"

# Gemini model for image generation (must be an image-capable model)
GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image"
gemini_client = genai.Client()

# --- System prompts ---

MINIMAL_TEXT_SYSTEM = """
You are an expert technical communicator. Given a topic, write a rich, detailed description 
that captures the key concepts, relationships, and flow of the system or idea.
Include:
- The main components or actors involved
- How they interact or connect
- The sequence or flow of operations
- Any important details that would help someone visualize the concept
Write 3-5 detailed sentences. Be specific and descriptive, not abstract.
"""

REVIEW_SYSTEM = """
You review technical descriptions for completeness and visual clarity.
Your goal is to ENRICH the text, not minimize it.
Check that the description includes:
1. All key components are named and described
2. Relationships and data flows are explicit
3. The sequence of operations is clear
4. Enough visual detail for someone to draw a diagram from it

If anything is missing or vague, add it. Return the improved, detailed version.
Always return at least 4-6 sentences of rich description.
"""

IMAGE_SYSTEM = """
You generate rich, visually detailed image prompts from approved text.
Take the template and fill in the placeholder with vivid visual details from the approved text.
For each component, suggest a specific ICON or VISUAL METAPHOR (e.g., a gear for processing, 
a brain for AI, a cloud for cloud services, an envelope for messaging, a shield for security,
a magnifying glass for search, a database cylinder for storage).
Make the scene feel alive with small decorative details: dotted lines, small stars, 
exclamation marks, lightbulbs, checkmarks, or small doodles around the main elements.
Your output MUST be under 900 characters total.
Return only the final image prompt, nothing else.
"""

IMAGE_PROMPT_TEMPLATE = """
16:9 landscape hand-drawn whiteboard illustration on off-white paper with faint blueprint grid lines.
COMPOSITION: All elements centered with 15% margin on all sides.

Subject: {approved_text}

VISUAL REQUIREMENTS:
- Each component represented by a DISTINCTIVE ICON (gears, brains, clouds, envelopes, shields, etc.) inside or above its labeled box
- Boxes connected by sketchy arrows with small annotations on the arrows describing the data flow
- Small decorative doodles: stars, lightbulbs, checkmarks, dotted trails, tiny sparkles
- Color-coded watercolor fills: each box a different soft color (blue, green, orange, coral, purple)
- Short Spanish labels in handwritten script inside each box

Style: hand-drawn pen-and-ink sketch with loose cross-hatching and watercolor accents. 
Analog, warm, textured feel — like a creative brainstorming whiteboard. No digital vectors. No title.
"""

RESTYLE_PROMPT = """
Recreate this image as a 16:9 landscape hand-drawn whiteboard illustration on off-white paper with faint blueprint grid lines.

TEXT RULES:
- Translate ONLY descriptive text and labels to Spanish (e.g., "Input" → "Entrada", "Process" → "Proceso")
- DO NOT translate proper names, brand names, product names — keep them EXACTLY as they appear
- DO NOT translate technical acronyms (API, SDK, HTTP, REST, etc.)
- When in doubt, keep the original text unchanged

LOGO RULES:
- Leave logos UNTOUCHED — same shape, same original colors, no modifications
- Do NOT add any text near or around logos
- Do NOT recolor logos with watercolor or any other fill
- Do NOT invent text to describe or label what a logo is

Keep the same concepts, components, and relationships shown in the original image, but redraw NON-LOGO elements in this style:
- Hand-drawn pen-and-ink sketch with loose cross-hatching
- Subtle watercolor color accents (blue, green, orange, coral, purple) for boxes and arrows only
- Small decorative doodles: stars, lightbulbs, checkmarks, dotted trails
- Handwritten script for labels
- Analog, warm, textured feel — like a creative brainstorming whiteboard
- No digital vectors, no title header
- All elements centered with 15% margin on all sides
"""

# --- Text Agents (OpenAI Agents SDK) ---

minimal_text_agent = Agent(
    name="minimal_text_agent",
    model=MODEL_TEXT,
    instructions=MINIMAL_TEXT_SYSTEM,
)

review_agent = Agent(
    name="review_agent",
    model=MODEL_REVIEW,
    instructions=REVIEW_SYSTEM,
)

image_prompt_agent = Agent(
    name="image_prompt_agent",
    model=MODEL_IMAGE_PROMPT,
    instructions=IMAGE_SYSTEM,
)

print(f"Text agents: OpenAI ({MODEL_TEXT}, {MODEL_REVIEW}, {MODEL_IMAGE_PROMPT})")
print(f"Image generation: Gemini ({GEMINI_IMAGE_MODEL})")

In [ ]:
import tempfile
import traceback
from pathlib import Path
from PIL import Image as PILImage
import gradio as gr

last_image_path = None

IMAGE_GEN_CONFIG = types.GenerateContentConfig(
    response_modalities=["IMAGE"],
    image_config=types.ImageConfig(aspect_ratio="16:9"),
)


# --- Tab 1: Generate from topic ---

async def generate_image_from_topic(message, history):
    global last_image_path

    try:
        yield "Generando texto inicial (OpenAI)..."
        proposal = (await Runner.run(minimal_text_agent, input=message)).final_output
        yield f"**Propuesta de texto:**\n{proposal}\n\nRevisando (OpenAI)..."

        approved_text = (await Runner.run(review_agent, input=proposal)).final_output
        yield f"**Texto aprobado:**\n{approved_text}\n\nGenerando prompt para imagen (OpenAI)..."

        image_input = IMAGE_PROMPT_TEMPLATE.format(approved_text=approved_text)
        image_prompt = (await Runner.run(image_prompt_agent, input=image_input)).final_output
        yield f"**Prompt de imagen:**\n{image_prompt}\n\nGenerando imagen (Gemini {GEMINI_IMAGE_MODEL})..."

        image_response = gemini_client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=image_prompt,
            config=IMAGE_GEN_CONFIG,
        )

        image_bytes = None
        for part in image_response.candidates[0].content.parts:
            if part.inline_data and part.inline_data.data:
                image_bytes = part.inline_data.data
                break

        if not image_bytes:
            yield "**Error:** No se generó imagen en la respuesta de Gemini."
            return

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        tmp.write(image_bytes)
        tmp.close()
        last_image_path = tmp.name

        yield gr.Image(tmp.name)

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


# --- Tab 2: Restyle pasted/uploaded image ---

def restyle_image(message, history):
    global last_image_path

    try:
        text = message.get("text", "").strip()
        files = message.get("files", [])

        if not files:
            yield "Pega o sube una imagen para regenerar. Puedes añadir instrucciones adicionales como texto."
            return

        image_path = files[0]
        image_bytes = Path(image_path).read_bytes()

        prompt = RESTYLE_PROMPT
        if text:
            prompt += f"\n\nAdditional instructions: {text}"

        yield "Regenerando imagen en estilo whiteboard 16:9 (Gemini)..."

        response = gemini_client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=[
                types.Content(
                    parts=[
                        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
                        types.Part.from_text(text=prompt),
                    ]
                )
            ],
            config=IMAGE_GEN_CONFIG,
        )

        result_bytes = None
        for part in response.candidates[0].content.parts:
            if part.inline_data and part.inline_data.data:
                result_bytes = part.inline_data.data
                break

        if not result_bytes:
            yield "**Error:** No se generó imagen en la respuesta de Gemini."
            return

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        tmp.write(result_bytes)
        tmp.close()
        last_image_path = tmp.name

        yield gr.Image(tmp.name)

    except Exception as e:
        yield f"**Error:**\n```\n{type(e).__name__}: {e}\n```\n\n```\n{traceback.format_exc()}\n```"


def download_last_image():
    return last_image_path


# --- UI ---

with gr.Blocks() as demo:
    gr.Markdown("# Image Generator Pipeline")
    gr.Markdown("Text agents: **OpenAI** | Image generation: **Gemini**")

    with gr.Tabs():
        with gr.TabItem("Generar desde tema"):
            gr.ChatInterface(
                fn=generate_image_from_topic,
                examples=["How multi-agent systems coordinate tasks using handoffs"],
                multimodal=False,
            )

        with gr.TabItem("Regenerar imagen"):
            gr.Markdown("Pega (Ctrl+V) o sube una imagen y se regenerará en estilo whiteboard con textos en español.")
            gr.ChatInterface(
                fn=restyle_image,
                multimodal=True,
            )

    download_btn = gr.DownloadButton("Descargar última imagen", variant="primary")
    download_btn.click(fn=download_last_image, outputs=download_btn)

demo.launch(inline=True)

# Visualitzar gasto

- OpenAI: https://platform.openai.com/usage
- Google: https://aistudio.google.com/spend